#Imports


In [22]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
#is to display plots directly below the code cell that generates them within the notebook interface. Without this line, you would typically need to manually type plt.show()
import keras
from keras.models import Sequential
from keras.layers import Dense, Dropout

from sklearn.metrics import confusion_matrix
import seaborn as sns
np.random.seed(0)
#y setting the seed to a specific number (like 0), you ensure that every time you run your code, the "random" calculations—such as initializing your model's weights—produce the exact same result.

#Data


In [9]:
from keras.datasets import mnist
(x_train, y_train), (x_test, y_test) = mnist.load_data()
#

In [10]:
print(x_train.shape , y_train.shape)
print(x_test.shape , y_test.shape)


(60000, 28, 28) (60000,)
(10000, 28, 28) (10000,)


 one-hot encoding for the labels.

Instead of representing labels as a single number (which might cause the network to incorrectly interpret values as continuous, e.g., predicting 4.5 for an image between 4 and 5), the code converts them into a vector of length 10.

Structure: This vector contains all zeros, except for a "1" at the index corresponding to the actual digit.
Purpose: This transformation enables the neural network to output the probability of an image belonging to each specific class, which is standard practice for classification tasks.

In [12]:
y_train = keras.utils.to_categorical(y_train, 10)
y_test = keras.utils.to_categorical(y_test,10 )

one-hot encoding is like a multiple-choice answer key.

Instead of telling the computer that a picture is a "number 5" (which might confuse it into thinking it's something between 4 and 6), we give it 10 separate boxes (labeled 0 to 9).

When the computer looks at an image, it puts a 1 in the correct box and a 0 in all the others.
This makes it very clear to the computer exactly which digit it is looking at, which helps it learn to be accurate.

In [13]:
for i in range (10):
  print (y_train[i])

[0. 0. 0. 0. 0. 1. 0. 0. 0. 0.]
[1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 1. 0. 0. 0. 0. 0.]
[0. 1. 0. 0. 0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0. 0. 0. 0. 1.]
[0. 0. 1. 0. 0. 0. 0. 0. 0. 0.]
[0. 1. 0. 0. 0. 0. 0. 0. 0. 0.]
[0. 0. 0. 1. 0. 0. 0. 0. 0. 0.]
[0. 1. 0. 0. 0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 1. 0. 0. 0. 0. 0.]


#Prepare data


In [15]:
#normalize data
x_train = x_train / 255.0
x_test = x_test / 255.0

In [17]:
#Reshaping the data into 1D vector to pass it (flattening)
x_train = x_train.reshape(x_train.shape[0], -1)
x_test = x_test.reshape(x_test.shape[0], -1)

In [18]:
print(x_train.shape)

(60000, 784)


# The actual mlp

In [20]:
model = Sequential()
# Sequential() is the way of adding multiple layers in keras
model.add(Dense(128, input_shape=(784,), activation='relu'))
model.add(Dense(128, activation='relu'))
model.add(Dropout(0.25))
model.add(Dense(10, activation='softmax'))

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │       100,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 118,282 (462.04 KB)

 Trainable params: 118,282 (462.04 KB)

 Non-trainable params: 0 (0.00 B)

This code builds a neural network that takes **784 input values** and predicts **one of 10 classes**—for example, recognizing digits from 0 to 9.

```python
model = Sequential()
```

Creates an empty model where layers run **in order**. Each layer’s output becomes the next layer’s input.

### 1. First hidden layer

```python
model.add(Dense(128, input_shape=(784,), activation='relu'))
```

- Receives **784 values** per sample, such as a flattened 28 × 28 image.
- Contains **128 neurons**, each connected to all 784 inputs.
- Uses **ReLU**: negative values become zero; positive values stay unchanged.
- Produces **128 values** that represent patterns learned from the input.

### 2. Second hidden layer

```python
model.add(Dense(128, activation='relu'))
```

Adds another layer of **128 neurons**.

It receives the 128 outputs from the previous layer and learns further combinations of those patterns. You don’t need to specify `input_shape` again because Keras infers it.

### 3. Dropout

```python
model.add(Dropout(0.25))
```

During training, randomly sets **25% of the incoming values to zero** at each step.

This helps reduce **overfitting**, where the model memorizes training examples but performs poorly on new examples. Dropout is turned off when evaluating or making predictions.

### 4. Output layer

```python
model.add(Dense(10, activation='softmax'))
```

Creates **10 output neurons**, one for each class.

**Softmax** turns their results into probabilities that sum to 1. For example:

```text
Digit 0: 0.01
Digit 1: 0.02
Digit 2: 0.90
Other digits combined: 0.07
```

The model predicts **digit 2**, since it has the highest probability.

### 5. Configure training

```python
model.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)
```

`compile()` configures how the model will learn; **it does not train it yet**.

| Setting | Meaning |
|---|---|
| `loss='categorical_crossentropy'` | Measures how far the predicted probabilities are from the correct class. Lower is better. |
| `optimizer='adam'` | Adjusts the weights and biases during training to reduce the loss. |
| `metrics=['accuracy']` | Reports the proportion of predictions that are correct. |

**Label format matters:** `categorical_crossentropy` expects **one-hot labels**. For digit 2, that looks like:

```python
[0, 0, 1, 0, 0, 0, 0, 0, 0, 0]
```

If your labels are integers such as `2`, use `sparse_categorical_crossentropy` instead.

### 6. Display the model

```python
model.summary()
```

Prints the layers, their output shapes, and their parameter counts:

| Layer | Output shape | Trainable parameters |
|---|---|---:|
| First Dense | `(None, 128)` | 100,480 |
| Second Dense | `(None, 128)` | 16,512 |
| Dropout | `(None, 128)` | 0 |
| Output Dense | `(None, 10)` | 1,290 |
| **Total** | | **118,282** |

**`None` represents the batch size**, which can vary. Dense parameters are calculated as:

\[
(\text{number of inputs}\times\text{number of neurons})+\text{number of biases}
\]

Dropout has zero parameters because it has no weights or biases to learn.

The complete flow is **784 inputs → 128 neurons → 128 neurons → dropout → 10 class probabilities**. Training starts when you call `model.fit(...)`.
# Parameters are the weights and biases learned during training.
# Dense parameters = (inputs * neurons) + neurons (one bias per neuron).
# First Dense: 784 * 128 + 128 = 100,480
# Second Dense: 128 * 128 + 128 = 16,512
# Output Dense: 128 * 10 + 10 = 1,290
# Total: 118,282; Dropout, ReLU, and softmax add no trainable parameters.
#
# Overfitting: the model performs well on training data but poorly on unseen data.
# Common sign: training loss decreases while validation loss increases.
# Reduce it using more varied data, dropout, early stopping, fewer neurons/layers,
# weight regularization, and suitable data augmentation.

#Train

In [25]:
batch_size = 512
epochs =10
model.fit(x= x_train, y=y_train, batch_size=batch_size, epochs=epochs)

Epoch 1/10
118/118 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.9932 - loss: 0.0221
Epoch 2/10
118/118 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.9940 - loss: 0.0196
Epoch 3/10
118/118 ━━━━━━━━━━━━━━━━━━━━ 3s 18ms/step - accuracy: 0.9947 - loss: 0.0185
Epoch 4/10
118/118 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - accuracy: 0.9949 - loss: 0.0169
Epoch 5/10
118/118 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - accuracy: 0.9949 - loss: 0.0164
Epoch 6/10
118/118 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.9954 - loss: 0.0157
Epoch 7/10
118/118 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.9958 - loss: 0.0141
Epoch 8/10
118/118 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.9961 - loss: 0.0136
Epoch 9/10
118/118 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - accuracy: 0.9964 - loss: 0.0128
Epoch 10/10
118/118 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - accuracy: 0.9965 - loss: 0.0120


#Evaluate


In [29]:
test_loss , test_acc = model.evaluate(x_test, y_test)
print("Test loss : {}, Test accuracy : {}".format(test_loss, test_acc))

313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.9820 - loss: 0.0658
Test loss : 0.06578055024147034, Test accuracy : 0.9819999933242798
